# Session 2: Building with LLMs - Companion Notebook

**Hands-on code demos** for Session 2 of the AI Engineering Teaching Series.

See `session_2_building_with_llms.md` for detailed theory.

## Learning Objectives

By the end of this notebook, you will:
- Apply prompt engineering techniques: zero-shot, few-shot, chain-of-thought
- Extract structured JSON from LLM responses with validation
- Chunk documents using different strategies and compare results
- Build keyword search and semantic search side by side
- Use ChromaDB as a vector database with metadata filtering
- Assemble a complete RAG pipeline from scratch
- Compare grounded vs ungrounded LLM responses

## Table of Contents

1. [Setup](#setup)
2. [Prompt Engineering](#prompts)
3. [Structured Outputs](#structured)
4. [Chunking Strategies](#chunking)
5. [Keyword vs Semantic Search](#search)
6. [ChromaDB: Vector Database](#chromadb)
7. [Full RAG Pipeline](#rag)
8. [Grounded vs Ungrounded](#grounding)
9. [Exercises](#exercises)

---
## 1. Setup <a id='setup'></a>

In [ ]:
# Setup
import os
import sys
import json
import numpy as np
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent.parent / ".env")

from openai import OpenAI
client = OpenAI()

print("Setup complete!")
print(f"OpenAI key: {'found' if os.getenv('OPENAI_API_KEY') else 'NOT FOUND - add to .env'}")

---
## 2. Prompt Engineering <a id='prompts'></a>

Comparing zero-shot, few-shot, and chain-of-thought on the same task: classifying customer support tickets.

In [ ]:
# Task: classify support tickets into categories
tickets = [
    "I can't log into my account, it says password incorrect",
    "When will my order #12345 arrive?",
    "Your app crashes every time I open the settings page",
    "I'd like a refund for my last purchase",
    "How do I change my email address on my profile?"
]

categories = ["account", "shipping", "bug", "billing", "account"]

# --- Zero-shot: just ask directly ---
print("=== Zero-Shot ===")
for ticket in tickets[:3]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Classify this support ticket into one category (account, shipping, bug, billing): '{ticket}'"}],
        temperature=0,
        max_tokens=20
    )
    print(f"  '{ticket[:50]}...' -> {response.choices[0].message.content.strip()}")

In [ ]:
# --- Few-shot: provide examples ---
print("=== Few-Shot ===")
few_shot_prompt = """Classify support tickets into: account, shipping, bug, billing.

Examples:
"I forgot my password" -> account
"My package hasn't arrived" -> shipping
"The page shows an error 500" -> bug
"I was charged twice" -> billing

Now classify:"""

for ticket in tickets[:3]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f'{few_shot_prompt}\n"{ticket}" ->'}],
        temperature=0,
        max_tokens=20
    )
    print(f"  '{ticket[:50]}...' -> {response.choices[0].message.content.strip()}")

In [ ]:
# --- Chain-of-thought: reason before answering ---
print("=== Chain-of-Thought ===")
cot_prompt = """Classify this support ticket. Think step by step:
1. What is the user's core issue?
2. Which category fits best: account, shipping, bug, billing?
3. Give your final classification.

Ticket: "{ticket}""""

ticket = "Your app crashes every time I open the settings page"
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": cot_prompt.format(ticket=ticket)}],
    temperature=0,
    max_tokens=150
)
print(response.choices[0].message.content)

---
## 3. Structured Outputs <a id='structured'></a>

Getting reliable JSON from LLMs using JSON mode and Pydantic validation.

In [ ]:
# JSON mode: extract structured data from unstructured text
from pydantic import BaseModel, ValidationError

# Define the schema we want
class ContactInfo(BaseModel):
    name: str
    email: str | None = None
    phone: str | None = None
    company: str | None = None

# Ask the model to extract info
text = "Hi, I'm Sarah Chen from TechCorp. You can reach me at sarah@techcorp.com or call 555-0123."

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract contact information as JSON with fields: name, email, phone, company. Use null for missing fields."},
        {"role": "user", "content": text}
    ],
    response_format={"type": "json_object"},
    temperature=0
)

raw_json = json.loads(response.choices[0].message.content)
print("Raw JSON from model:")
print(json.dumps(raw_json, indent=2))

# Validate with Pydantic
try:
    contact = ContactInfo(**raw_json)
    print(f"\nValidated: {contact.name} at {contact.company}, email: {contact.email}")
except ValidationError as e:
    print(f"Validation failed: {e}")

In [ ]:
# Batch extraction: process multiple texts
texts = [
    "Meeting with John Doe (john@example.com) from Acme Inc tomorrow.",
    "Please forward to Dr. Lisa Park, phone 555-9876.",
    "Contact: mike.jones@startup.io, CTO of DataFlow Labs"
]

results = []
for text in texts:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract contact info as JSON: name, email, phone, company. Use null for missing fields."},
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )
    data = json.loads(response.choices[0].message.content)
    contact = ContactInfo(**data)
    results.append(contact)

print(f"{'Name':<20} {'Email':<30} {'Phone':<15} {'Company':<20}")
print("-" * 85)
for c in results:
    print(f"{c.name:<20} {str(c.email):<30} {str(c.phone):<15} {str(c.company):<20}")

---
## 4. Chunking Strategies <a id='chunking'></a>

Comparing fixed-size, recursive, and token-based chunking on the same document.

In [ ]:
# Sample document to chunk
document = """
# Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that enables systems to learn from data. Instead of being explicitly programmed, these systems improve through experience.

## Supervised Learning

In supervised learning, the model learns from labeled examples. Given input-output pairs, it learns a mapping function. Common algorithms include linear regression, decision trees, and neural networks.

For example, a spam classifier is trained on thousands of emails labeled as "spam" or "not spam." After training, it can classify new, unseen emails.

## Unsupervised Learning

Unsupervised learning works with unlabeled data. The model discovers hidden patterns and structures. Clustering algorithms like K-means group similar data points together.

Principal Component Analysis (PCA) reduces dimensionality while preserving the most important information in the data.

## Deep Learning

Deep learning uses neural networks with many layers. These networks can learn hierarchical representations, from simple features in early layers to complex concepts in deeper layers.

Convolutional Neural Networks (CNNs) excel at image tasks. Recurrent Neural Networks (RNNs) handle sequential data. Transformers have revolutionized natural language processing.
"""

# --- Fixed-size chunking ---
def fixed_chunk(text, size=300, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end].strip())
        start = end - overlap
    return [c for c in chunks if c]

# --- Recursive chunking ---
def recursive_chunk(text, size=300, overlap=50):
    # Split by double newline (paragraphs) first
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    
    chunks = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) + 2 <= size:
            current = current + "\n\n" + para if current else para
        else:
            if current:
                chunks.append(current)
            current = para
    if current:
        chunks.append(current)
    return chunks

print("=== Fixed-Size Chunking (300 chars) ===")
fixed = fixed_chunk(document, size=300)
for i, chunk in enumerate(fixed[:3]):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(f"  '{chunk[:80]}...'")

print(f"\nTotal: {len(fixed)} chunks")

print("\n=== Recursive Chunking (300 chars) ===")
recursive = recursive_chunk(document, size=300)
for i, chunk in enumerate(recursive[:3]):
    print(f"\nChunk {i+1} ({len(chunk)} chars):")
    print(f"  '{chunk[:80]}...'")

print(f"\nTotal: {len(recursive)} chunks")
print("\nNotice: recursive chunking respects paragraph boundaries.")

---
## 5. Keyword vs Semantic Search <a id='search'></a>

Side-by-side comparison showing where each approach wins and fails.

In [ ]:
# Corpus of documents
corpus = [
    "Python is a versatile programming language used for web development, data science, and AI.",
    "The canine jumped over the fence and ran across the yard.",
    "Error code 404 means the requested page was not found on the server.",
    "Machine learning algorithms learn patterns from historical data.",
    "My puppy loves playing fetch in the park every morning.",
    "HTTP status codes indicate whether a request was successful or failed.",
    "Neural networks are inspired by biological brain structures.",
    "The dog barked loudly at the mail carrier approaching the house.",
    "Server responded with a 404 Not Found error for the API endpoint.",
    "Deep learning models require large amounts of training data and compute."
]

# --- Keyword search (simple TF-IDF style) ---
from collections import Counter
import math

def keyword_search(query, docs, top_k=3):
    """Simple BM25-style keyword search."""
    query_terms = query.lower().split()
    scores = []
    for doc in docs:
        doc_lower = doc.lower()
        score = sum(doc_lower.count(term) for term in query_terms)
        # Bonus for exact phrase fragments
        if query.lower() in doc_lower:
            score += 5
        scores.append(score)
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [(corpus[i], s) for i, s in ranked[:top_k] if s > 0]

# --- Semantic search ---
def get_embedding(text):
    response = client.embeddings.create(model="text-embedding-3-small", input=text)
    return response.data[0].embedding

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Pre-embed corpus
corpus_embeddings = [get_embedding(doc) for doc in corpus]

def semantic_search(query, top_k=3):
    query_emb = get_embedding(query)
    scores = [(i, cosine_sim(query_emb, emb)) for i, emb in enumerate(corpus_embeddings)]
    scores.sort(key=lambda x: x[1], reverse=True)
    return [(corpus[i], s) for i, s in scores[:top_k]]

# Compare on different queries
queries = [
    "dog",                    # Semantic should find "canine", "puppy"
    "error code 404",         # Keyword should find exact matches
    "how do machines learn?", # Semantic should understand the meaning
]

for query in queries:
    print(f"Query: '{query}'")
    print(f"  Keyword results:")
    for doc, score in keyword_search(query, corpus):
        print(f"    [{score:.0f}] {doc[:65]}...")
    print(f"  Semantic results:")
    for doc, score in semantic_search(query):
        print(f"    [{score:.3f}] {doc[:65]}...")
    print()

---
## 6. ChromaDB: Vector Database <a id='chromadb'></a>

A real vector database with metadata filtering.

In [ ]:
# ChromaDB: create collection, add documents, query
import chromadb

chroma_client = chromadb.Client()

# Create a collection
collection = chroma_client.create_collection(
    name="session2_demo",
    metadata={"hnsw:space": "cosine"}
)

# Knowledge base with metadata
docs = [
    {"text": "Python is used for web development with Django and Flask frameworks.", "category": "programming", "difficulty": "beginner"},
    {"text": "React and Vue are popular JavaScript frameworks for building user interfaces.", "category": "programming", "difficulty": "intermediate"},
    {"text": "RAG combines retrieval with generation for more accurate AI responses.", "category": "ai", "difficulty": "intermediate"},
    {"text": "Fine-tuning adapts a pre-trained model to your specific task or domain.", "category": "ai", "difficulty": "advanced"},
    {"text": "Docker containers package applications with all dependencies for consistent deployment.", "category": "devops", "difficulty": "intermediate"},
    {"text": "Kubernetes orchestrates container deployment, scaling, and management.", "category": "devops", "difficulty": "advanced"},
    {"text": "PostgreSQL is a powerful open-source relational database system.", "category": "databases", "difficulty": "beginner"},
    {"text": "Vector databases like ChromaDB enable fast similarity search over embeddings.", "category": "databases", "difficulty": "intermediate"},
]

collection.add(
    documents=[d["text"] for d in docs],
    ids=[f"doc_{i}" for i in range(len(docs))],
    metadatas=[{"category": d["category"], "difficulty": d["difficulty"]} for d in docs]
)

print(f"Added {collection.count()} documents to ChromaDB")

# Basic query
results = collection.query(query_texts=["How do I build AI applications?"], n_results=3)
print("\nQuery: 'How do I build AI applications?'")
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"  [{1-dist:.3f}] {doc}")

In [ ]:
# Query with metadata filter
print("=== Filter: only 'ai' category ===")
results = collection.query(
    query_texts=["What techniques improve model performance?"],
    n_results=3,
    where={"category": "ai"}
)
for doc in results["documents"][0]:
    print(f"  {doc}")

print("\n=== Filter: only 'beginner' difficulty ===")
results = collection.query(
    query_texts=["What tools should I learn first?"],
    n_results=3,
    where={"difficulty": "beginner"}
)
for doc in results["documents"][0]:
    print(f"  {doc}")

---
## 7. Full RAG Pipeline <a id='rag'></a>

End-to-end: load documents, chunk, embed, store, retrieve, generate.

In [ ]:
# Build a complete RAG system from scratch
import chromadb
from openai import OpenAI

client = OpenAI()
chroma_client = chromadb.Client()

# Step 1: Our "knowledge base" (in practice, load from files)
knowledge_base = [
    "Acme Corp was founded in 2015 by Jane Smith and operates in the enterprise software space.",
    "Acme Corp's flagship product is AcmeAI, a platform for building custom AI applications.",
    "AcmeAI supports Python and JavaScript SDKs with full API documentation.",
    "Pricing: AcmeAI offers a free tier (1000 API calls/month), Pro ($49/month, 50K calls), and Enterprise (custom pricing).",
    "Acme Corp's refund policy: full refund within 14 days of purchase. After 14 days, store credit only.",
    "AcmeAI integrates with AWS, GCP, and Azure cloud platforms.",
    "The AcmeAI Python SDK requires Python 3.9+ and can be installed with pip install acmeai.",
    "Acme Corp's support hours are Monday-Friday 9am-6pm EST. Email: support@acmecorp.com.",
    "AcmeAI uses end-to-end encryption and is SOC 2 Type II certified.",
    "Common issues: if you get a 429 error, you've exceeded your rate limit. Upgrade your plan or add a retry with exponential backoff."
]

# Step 2: Chunk (documents are already short, so each is one chunk)
# In production, you'd chunk longer documents here

# Step 3: Store in vector DB
rag_collection = chroma_client.create_collection(name="acme_rag")
rag_collection.add(
    documents=knowledge_base,
    ids=[f"kb_{i}" for i in range(len(knowledge_base))]
)
print(f"Indexed {rag_collection.count()} documents")

# Step 4: RAG query function
def rag_query(question, top_k=3):
    # Retrieve
    results = rag_collection.query(query_texts=[question], n_results=top_k)
    context = "\n\n".join(results["documents"][0])
    
    # Generate
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant for Acme Corp. Answer questions using ONLY the provided context. If the context doesn't contain the answer, say 'I don't have that information.' Always cite which part of the context supports your answer."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        temperature=0
    )
    return {
        "answer": response.choices[0].message.content,
        "sources": results["documents"][0]
    }

# Test it
questions = [
    "What is Acme Corp's refund policy?",
    "How much does AcmeAI Pro cost?",
    "What should I do if I get a 429 error?",
]

for q in questions:
    result = rag_query(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"Sources used: {len(result['sources'])}")
    print()

---
## 8. Grounded vs Ungrounded <a id='grounding'></a>

The same question with and without context: see the difference.

In [ ]:
# Ungrounded: ask without context (model will make things up)
print("=== UNGROUNDED (no context) ===\n")
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What is Acme Corp's refund policy? Be specific about timeframes and conditions."}
    ],
    temperature=0
)
print(f"A: {response.choices[0].message.content}")

# Grounded: ask with RAG context
print("\n=== GROUNDED (with RAG context) ===\n")
result = rag_query("What is Acme Corp's refund policy? Be specific about timeframes and conditions.")
print(f"A: {result['answer']}")

print("\n--- Comparison ---")
print("The ungrounded answer sounds plausible but is fabricated.")
print("The grounded answer is accurate because it draws from real documents.")

---
## 9. Exercises <a id='exercises'></a>

### Exercise 1: Build a RAG System on Your Own Documents

Create a knowledge base of at least 10 documents about a topic you know well 
(your company's FAQ, a hobby, a technical domain). Build a RAG pipeline and test 
with 5 questions. Verify the answers are grounded in your documents.

In [ ]:
# Exercise 1: Your code here

# Step 1: Define your knowledge base (at least 10 documents)
my_knowledge_base = [
    # TODO: add your documents
]

# Step 2: Index in ChromaDB

# Step 3: Build a rag_query function

# Step 4: Test with 5 questions and verify accuracy

### Exercise 2: Experiment with Chunk Sizes

Take a long document (at least 2000 characters). Chunk it at sizes 200, 500, and 1000 characters.
For each chunk size, run 3 test queries and compare which chunk size gives the best retrieval results.

In [ ]:
# Exercise 2: Your code here

long_document = """
# TODO: paste a long document here (at least 2000 chars)
"""

# Chunk at different sizes
# Index each set of chunks in a separate ChromaDB collection
# Run the same 3 queries against each
# Compare: which chunk size retrieves the most relevant content?

### Exercise 3: Add Hybrid Search to the RAG Pipeline

Combine keyword search (matching exact terms) with semantic search (ChromaDB).
For each query, run both searches, merge the results, and use the combined context for generation.
Test on queries where keyword search wins and where semantic search wins.

In [ ]:
# Exercise 3: Your code here

def hybrid_rag_query(question, top_k=3):
    # TODO:
    # 1. Run keyword search over knowledge_base
    # 2. Run semantic search via ChromaDB
    # 3. Merge results (remove duplicates, combine top results)
    # 4. Generate answer from merged context
    pass

# Test with queries that benefit from each approach:
# "error 429" (keyword wins: exact match)
# "how to get my money back" (semantic wins: meaning match for "refund")

---
## Checkpoint

Before moving on to Session 3, verify:

- [ ] You can apply zero-shot, few-shot, and chain-of-thought prompting
- [ ] You can extract structured JSON and validate with Pydantic
- [ ] You understand the tradeoffs between chunking strategies
- [ ] You've seen where keyword search wins and where semantic search wins
- [ ] You can use ChromaDB with metadata filtering
- [ ] You built a complete RAG pipeline and tested it
- [ ] You understand why grounding matters (grounded vs ungrounded comparison)

### Next: Session 3 - Agentic AI

We'll build AI agents that can reason, plan, and take actions using tools.